In [ ]:
from src.ingestion.loader import AmazonReviewLoader

loader = AmazonReviewLoader(
    "datasets/raw/reviews_Tools_and_Home_Improvement.json.gz"
)

for i, review in enumerate(loader.stream()):

    print(review)

    if i == 2:
        break

In [ ]:
from src.ingestion.loader import AmazonReviewLoader
from src.ingestion.mapper import AmazonReviewMapper


loader = AmazonReviewLoader(
    "datasets/raw/reviews_Tools_and_Home_Improvement.json.gz"
)

record = next(loader.stream())

mapped = AmazonReviewMapper.map(record)

for k, v in mapped.items():
    print(f"{k:20} : {v}")

In [ ]:
from src.ingestion.loader import AmazonReviewLoader
from src.ingestion.mapper import AmazonReviewMapper
from src.ingestion.validator import RetailReviewValidator

loader = AmazonReviewLoader(
    "datasets/raw/reviews_Tools_and_Home_Improvement.json.gz"
)

for review in loader.stream():

    mapped = AmazonReviewMapper.map(review)

    valid, reason = RetailReviewValidator.validate(mapped)

    print(valid, reason)

    break

In [ ]:
from pprint import pprint

from src.ingestion.loader import AmazonReviewLoader
from src.ingestion.mapper import AmazonReviewMapper
from src.ingestion.validator import RetailReviewValidator
from src.ingestion.profiler import DataProfiler

loader = AmazonReviewLoader(
    "datasets/raw/reviews_Tools_and_Home_Improvement.json.gz"
)

profiler = DataProfiler()

for review in loader.stream():

    mapped = AmazonReviewMapper.map(review)

    valid, _ = RetailReviewValidator.validate(mapped)

    if valid:
        profiler.update(mapped)

pprint(profiler.report())

In [ ]:
from src.preprocessing.text_cleaner import TextCleaner
review = """
This was &#34;Amazing&#34;.

Visit https://amazon.com

Email me at abc@gmail.com
"""

print(TextCleaner.clean(review))

In [ ]:
from src.preprocessing.feature_engineering import FeatureEngineer
review = {
    "review_text": "This drill is amazing! Would definitely recommend it.",
    "helpful_yes": 8,
    "helpful_total": 10,
}

review = FeatureEngineer.generate(review)

for k, v in review.items():
    print(f"{k:<25} {v}")

In [ ]:
import gzip

with gzip.open(
    "datasets/raw/meta_Tools_and_Home_Improvement.json.gz",
    "rt",
    encoding="utf-8"
) as f:

    for i in range(3):
        print(f.readline())

In [ ]:
from src.ingestion.metadata_joiner import MetadataJoiner

review = {
    "product_id": "001212835X",
    "review_text": "Excellent lamp."
}

metadata = [
    {
        "product_id": "001212835X",
        "product_name": "Everett's Cottage Table Lamp",
        "category": "Lighting",
        "brand": "Everett",
        "image_url": "...",
        "description": "...",
        "features": []
    }
]

joiner = MetadataJoiner(metadata)

review = joiner.enrich(review)

print(review)

In [ ]:
from src.pipeline.retail_data_pipeline import RetailDataPipeline

pipeline = RetailDataPipeline(
    review_path="datasets/raw/reviews_Tools_and_Home_Improvement.json.gz",
    metadata_path="datasets/raw/meta_Tools_and_Home_Improvement.json.gz",
    output_path="datasets/processed/master_reviews.parquet",
)

df = pipeline.build()

print(df.head())

In [ ]:
from src.ingestion.meta_data_loader import MetadataLoader

loader = MetadataLoader(
    "datasets/raw/meta_Tools_and_Home_Improvement.json.gz"
)

for i, record in enumerate(loader.stream()):
    print(record.keys())

    print(record)

    if i == 2:
        break

In [ ]:
from src.ingestion.meta_data_loader import MetadataLoader

loader = MetadataLoader(
    "datasets/raw/meta_Tools_and_Home_Improvement.json.gz"
)

count = 0
total = 0

for record in loader.stream():

    total += 1

    if "feature" in record:
        count += 1

print("Products:", total)
print("Products with feature:", count)

In [ ]:
from src.ingestion.meta_data_loader import MetadataLoader

loader = MetadataLoader(
    "datasets/raw/meta_Tools_and_Home_Improvement.json.gz"
)

record = next(loader.stream())

print(record.keys())

In [ ]:
from pprint import pprint

from src.training.sentiment.tokenizer_analyzer import (
    TokenizerAnalyzer,
)

analyzer = TokenizerAnalyzer(
    "datasets/processed/master_reviews.parquet"
)

report = analyzer.analyze()

print("\n===== TOKEN REPORT =====")

pprint(report)

In [ ]:
from src.training.sentiment.data_splitter import DataSplitter

splitter = DataSplitter(
    "datasets/processed/master_reviews.parquet"
)

train_df, val_df, test_df = splitter.split(
    output_dir="datasets/splits"
)


In [ ]:
from transformers import AutoTokenizer

from src.training.sentiment.dataset import RetailSentimentDataset

tokenizer = AutoTokenizer.from_pretrained(
    "bert-base-uncased"
)

dataset = RetailSentimentDataset(
    parquet_path="datasets/splits/train.parquet",
    tokenizer=tokenizer,
    max_length=256,
)

sample = dataset[0]

print(sample["input_ids"].shape)
print(sample["attention_mask"].shape)
print(sample["label"])

In [ ]:
from transformers import AutoTokenizer

from src.training.sentiment.dataset import RetailSentimentDataset
from src.training.sentiment.dataloader import RetailDataLoaderFactory

tokenizer = AutoTokenizer.from_pretrained(
    "bert-base-uncased"
)

dataset = RetailSentimentDataset(
    parquet_path="datasets/splits/train.parquet",
    tokenizer=tokenizer,
    max_length=256,
)

loader = RetailDataLoaderFactory.create(
    dataset,
    batch_size=8,
    shuffle=True,
)

batch = next(iter(loader))

print(batch["input_ids"].shape)
print(batch["attention_mask"].shape)
print(batch["label"].shape)

In [ ]:
from src.training.sentiment.metrics import SentimentMetrics
import numpy as np

labels = np.array([0, 1, 2, 2, 1, 0])

predictions = np.array([0, 1, 2, 1, 1, 0])

metrics = SentimentMetrics.compute(
    labels,
    predictions,
)

print(metrics)

print(
    SentimentMetrics.classification_report(
        labels,
        predictions,
    )
)

print(
    SentimentMetrics.confusion_matrix(
        labels,
        predictions,
    )
)

In [ ]:
from pprint import pprint

from src.training.sentiment.class_weights import (
    ClassWeightCalculator,
)

weights = ClassWeightCalculator.compute(
    "datasets/splits/train.parquet"
)

pprint(weights)

In [ ]:
from src.training.device import get_device
from src.training.sentiment.class_weights import ClassWeightCalculator
from src.training.sentiment.losses import LossFactory

device = get_device()

weights = ClassWeightCalculator.compute_tensor(
    "datasets/splits/train.parquet",
    device=device,
)

print(weights)

criterion = LossFactory.cross_entropy(
    class_weights=weights
)

print(criterion)

In [ ]:
from transformers import AutoTokenizer

from src.training.sentiment.dataset import RetailSentimentDataset
from src.training.sentiment.dataloader import RetailDataLoaderFactory

from src.training.sentiment.trainer import RetailSentimentTrainer
from src.training.sentiment.optimizer import OptimizerFactory
from src.training.sentiment.class_weights import ClassWeightCalculator
from src.training.sentiment.losses import LossFactory

from src.training.sentiment.config import SentimentConfig

from src.training.device import get_device


device = get_device()

tokenizer = AutoTokenizer.from_pretrained(
    SentimentConfig.MODEL_NAME
)

dataset = RetailSentimentDataset(
    parquet_path=SentimentConfig.TRAIN_DATA,
    tokenizer=tokenizer,
    max_length=SentimentConfig.MAX_LENGTH,
)

loader = RetailDataLoaderFactory.create(
    dataset,
    batch_size=8,
    shuffle=True,
)

trainer = RetailSentimentTrainer(
    model_name=SentimentConfig.MODEL_NAME,
    num_labels=SentimentConfig.NUM_CLASSES,
)

weights = ClassWeightCalculator.compute_tensor(
    SentimentConfig.TRAIN_DATA,
    device,
)

criterion = LossFactory.cross_entropy(
    weights,
)

optimizer = OptimizerFactory.adamw(
    trainer.model,
    learning_rate=SentimentConfig.LEARNING_RATE,
    weight_decay=SentimentConfig.WEIGHT_DECAY,
)

batch = next(iter(loader))

loss = trainer.train_one_batch(
    batch,
    optimizer,
    criterion,
)

print(loss)

In [ ]:
from src.training.sentiment.model_factory import ModelFactory

from src.training.sentiment.config import (
    SentimentConfig,
)

model = ModelFactory.create(

    model_name=SentimentConfig.MODEL_NAME,

    num_labels=SentimentConfig.NUM_CLASSES,

    id2label=SentimentConfig.ID2LABEL,

    label2id=SentimentConfig.LABEL2ID,
)

print(type(model))

In [ ]:
import pandas as pd

train = pd.read_parquet("datasets/splits/train.parquet")
val = pd.read_parquet("datasets/splits/validation.parquet")

train.head(500).to_parquet(
    "datasets/splits/train_small.parquet",
    index=False,
)

val.head(200).to_parquet(
    "datasets/splits/validation_small.parquet",
    index=False,
)

print("Done")

from transformers import AutoTokenizer

from src.training.sentiment.dataset import RetailSentimentDataset
from src.training.sentiment.dataloader import RetailDataLoaderFactory
from src.training.sentiment.config import SentimentConfig

tokenizer = AutoTokenizer.from_pretrained(
    SentimentConfig.MODEL_NAME
)

train_dataset = RetailSentimentDataset(
    parquet_path="datasets/splits/train_small.parquet",
    tokenizer=tokenizer,
    max_length=SentimentConfig.MAX_LENGTH,
)

validation_dataset = RetailSentimentDataset(
    parquet_path="datasets/splits/validation_small.parquet",
    tokenizer=tokenizer,
    max_length=SentimentConfig.MAX_LENGTH,
)

train_loader = RetailDataLoaderFactory.create(
    train_dataset,
    batch_size=8,
    shuffle=True,
)

validation_loader = RetailDataLoaderFactory.create(
    validation_dataset,
    batch_size=8,
    shuffle=False,
)

from src.training.device import get_device

from src.training.sentiment.model_factory import ModelFactory
from src.training.sentiment.optimizer import OptimizerFactory
from src.training.sentiment.scheduler import SchedulerFactory
from src.training.sentiment.class_weights import (
    ClassWeightCalculator,
)
from src.training.sentiment.losses import LossFactory
from src.training.sentiment.trainer import (
    TransformerTrainer,
)

device = get_device()

model = ModelFactory.create(
    model_name=SentimentConfig.MODEL_NAME,
    num_labels=SentimentConfig.NUM_CLASSES,
    id2label=SentimentConfig.ID2LABEL,
    label2id=SentimentConfig.LABEL2ID,
)

model.to(device)

weights = ClassWeightCalculator.compute_tensor(
    "datasets/splits/train_small.parquet",
    device=device,
)

criterion = LossFactory.cross_entropy(weights)

optimizer = OptimizerFactory.adamw(
    model=model,
    learning_rate=SentimentConfig.LEARNING_RATE,
    weight_decay=SentimentConfig.WEIGHT_DECAY,
)

total_steps = (
    len(train_loader)
    * 1
)

scheduler = SchedulerFactory.linear_warmup(
    optimizer=optimizer,
    total_training_steps=total_steps,
    warmup_ratio=SentimentConfig.WARMUP_RATIO,
)

trainer = TransformerTrainer(
    model=model,
    optimizer=optimizer,
    scheduler=scheduler,
    criterion=criterion,
    tokenizer=tokenizer,
    device=device,
)

trainer.fit(
    train_loader=train_loader,
    validation_loader=validation_loader,
    epochs=1,
)

In [1]:
from pathlib import Path
from copy import deepcopy

from src.training.sentiment.config import SentimentConfig


class SmallConfig(SentimentConfig):

    TRAIN_DATA = Path("datasets/splits/train_small.parquet")

    VALIDATION_DATA = Path("datasets/splits/validation_small.parquet")

    TEST_DATA = Path("datasets/splits/test_small.parquet")

    BATCH_SIZE = 8

    NUM_EPOCHS = 1

from src.training.sentiment.train import main

main(
    resume=False,
    config=SmallConfig,
)

/Users/prakash/envs/retail/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-06-28 01:29:15,400 | INFO | Using device: mps
2026-06-28 01:29:15,401 | INFO | Loading tokenizer: bert-base-uncased
2026-06-28 01:29:15,660 | INFO | HTTP Request: HEAD https://huggingface.co/bert-base-uncased/resolve/main/config.json "HTTP/1.1 200 OK"
2026-06-28 01:29:15,903 | INFO | HTTP Request: HEAD https://huggingface.co/bert-base-uncased/resolve/main/tokenizer_config.json "HTTP/1.1 200 OK"
2026-06-28 01:29:16,134 | INFO | HTTP Request: GET https://huggingface.co/api/models/bert-base-uncased/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 307 Temporary Redirect"
2026-06-28 01:29:16,368 | INFO | HTTP Request: GET https://huggingface.co/api/models/google-bert/bert-base-uncased/tree/main/additiona


Epoch 1/1


2026-06-28 01:30:18,288 | INFO | Training Loss : 1.0926                                    
/Users/prakash/envs/retail/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/prakash/envs/retail/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/prakash/envs/retail/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter 


Train Loss      : 1.0926
Validation Loss : 1.1074
Accuracy         : 0.6500
Precision        : 0.3903
Recall           : 0.5019
Macro F1         : 0.4028
Learning Rate    : 0.00e+00


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.50it/s]
2026-06-28 01:30:34,470 | INFO | Checkpoint saved at artifacts/sentiment/best_model
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.75it/s]
2026-06-28 01:30:36,152 | INFO | Checkpoint saved at artifacts/sentiment/last_model
2026-06-28 01:30:36,159 | INFO | Training history saved.
2026-06-28 01:30:36,159 | INFO | Training Completed.



FINAL TEST EVALUATION


/Users/prakash/envs/retail/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/prakash/envs/retail/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/prakash/envs/retail/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result


Evaluation
Loss      : 1.0539
Accuracy  : 0.6500
Precision : 0.3685
Recall    : 0.4441
Macro F1  : 0.3741
